In [1]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn
import torch.optim as optim
from utils.SplitData import split_data_Train_Val_Test
from torch.utils.tensorboard import SummaryWriter # type: ignore
import time
import numpy as np
from model.trace import TRACE
from dataset.otto_trace import TraceOttoDataSet

from utils.EarlyStopping import EarlyStopping
from utils.feature_engineering import get_between_features, get_elapsed_feature


In [ ]:
#DataSet    
dataset_processed = TraceOttoDataSet(
    file_name='train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=16,
    max_samples=1000
)

      
#See if the Lenght of the new inputs are the Lenght Sequence
sample = dataset_processed[0]
assert len(sample[0]["timestamps"]) == 64


print("================================================ (Logits part) ===================================================")
print("Logits Data_set_processed the ATC (Add to the Cart 3 times)")
print(dataset_processed.__ATC_task_logit__())
    
print("Logits for SAT4 (Seeing the same Aid 4 times)")
print(dataset_processed.__SAT__task_logit__())
    
print("Logits for PD1 (Make any Purchase within a day)")
print(dataset_processed.__PD1_task_logit___())
    
print("Logits for RA1 (Return to the same Aid in 1 days)")
print(dataset_processed.__RA1_task_logit___())

================================================ (Logits part) ===================================================
Logits Data_set_processed the ATC (Add to the Cart 4 times)
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,

In [ ]:
train_loader, validation_loader, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=32)


In [ ]:
"""all_ATC = 0
all_SAT = 0
all_PD1 = 0
all_RA1 = 0
for idx, (inputs_val, targets_val) in enumerate(train_loader):
    all_ATC += targets_val['ATC'].sum()
    all_SAT += targets_val['SAT'].sum()
    all_PD1 += targets_val['PD1'].sum()
    all_RA1 += targets_val['RA1'].sum()
    print(targets_val['ATC'])
    #print(targets_val['PD1'])
    #print(targets_val['SAT'])
    #print(targets_val['RA1'])
    if idx > 5:
        break
print(all_ATC)
print(all_PD1)
print(all_SAT)
print(all_RA1)"""

In [ ]:
all_ATC = 0
all_SAT = 0
all_PD1 = 0
all_RA1 = 0
for inputs_val, targets_val in validation_loader:
    all_ATC += targets_val['ATC'].sum()
    all_SAT += targets_val['SAT'].sum()
    all_PD1 += targets_val['PD1'].sum()
    all_RA1 += targets_val['RA1'].sum()
    
print(all_ATC)
print(all_SAT)
print(all_PD1)
print(all_RA1)

In [ ]:
max_aid = max(
    session[0]["aid"].max().item()
    for session in dataset_processed
)
max_type = max(
    session[0]["type"].max().item()
    for session in dataset_processed
)

num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    embedding_dim=32,
    num_classes=4 
)

In [ ]:
"""
Shape of train_loader, val_loader, Test_loader
"""
for inputs_train, targets_train in train_loader:
    print("The version of train_loader")
    print(inputs_train.keys())
    print(targets_train.keys())
    print(
        f"Shape Aids: {inputs_train['aid'].shape}, "
        f"Shape Timestamps: {inputs_train['timestamps'].shape}, "
        f"Shape Type: {inputs_train['type'].shape}"
    )
    break  

for inputs_val, targets_val in validation_loader:
    
    print("The version of val_loader")
    print(inputs_val.keys())
    print(targets_val.keys())
    
    print(
        f"Shape Aids: {inputs_val['aid'].shape}, "
        f"Shape Timestamps: {inputs_val['timestamps'].shape}, "
        f"Shape Type: {inputs_val['type'].shape}"
    )
    break
for inputs_test, targets_test in test_loader:
    print("The version of train_loader")
    print(inputs_test.keys())
    print(targets_test.keys())
    print(
        f"Shape Aids: {inputs_test['aid'].shape}, "
        f"Shape Timestamps: {inputs_test['timestamps'].shape}, "
        f"Shape Type: {inputs_test['type'].shape}"
    )
    break

print("==================================SIZE=================================== ")
print(f"Train LOADER {len(train_loader)}")  
print(f"Validation LOADER {len(validation_loader)}")  
print(f"Test LOADER {len(test_loader)}")  


In [ ]:
trace_model = trace_model
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(trace_model.parameters(), lr=3e-5, weight_decay=1e-6)
early_stopping = EarlyStopping(
    patience=6,
    min_delta=1e-4,
    mode="min",
    path="best_TRACE_model.pt"
    )
num_epochs = 5

    #Summary Writer for tensorBoard
tensor_board_writer = SummaryWriter(log_dir=f"TensorBoard_runs/Debugging/exp_{time.time()}")
trace_model.train()

for epoch in range(num_epochs):
        # -------------------------------TRAINING ---------------------------
        trace_model.train()
        epoch_loss = 0.0

        correct_train_ATC = 0
        correct_train_SAT = 0
        correct_train_PD1 = 0
        correct_train_RA1 = 0

        total_train_ATC = 0
        total_train_SAT = 0
        total_train_PD1 = 0
        total_train_RA1 = 0

        for inputs_train, targets_train in train_loader:

           
            label_train_ATC = targets_train["ATC"].unsqueeze(1)
            label_train_SAT = targets_train["SAT"].unsqueeze(1)
            label_train_PD1 = targets_train["PD1"].unsqueeze(1)
            label_train_RA1 = targets_train["RA1"].unsqueeze(1)

            delta_elapsed = get_elapsed_feature(inputs_train["timestamps"])
            delta_between = get_between_features(inputs_train["timestamps"])

            logits_train = trace_model(
                inputs_train["aid"],
                inputs_train["type"],
                delta_elapsed,
                delta_between
            )

            pred_ATC = logits_train[:, 0:1] #ATC
            pred_SAT = logits_train[:, 1:2] #SAT
            pred_PD1 = logits_train[:, 2:3] #PD1
            pred_RA1 = logits_train[:, 3:4] #RA1

            loss_ATC = criterion(pred_ATC, label_train_ATC.float())
            loss_SAT = criterion(pred_SAT, label_train_SAT.float())
            loss_PD1 = criterion(pred_PD1, label_train_PD1.float())
            loss_RA1 = criterion(pred_RA1, label_train_RA1.float())

            optimizer.zero_grad()
            
            loss_training = loss_ATC + loss_SAT + loss_PD1 + loss_RA1
            
            loss_training.backward()
            
            optimizer.step()

            epoch_loss += loss_training.item()

            # ============ ATC ============
            probs_ATC = torch.sigmoid(pred_ATC)
            preds_ATC = (probs_ATC >= 0.5).float()
            correct_train_ATC += (preds_ATC == label_train_ATC).sum().item()
            total_train_ATC += label_train_ATC.numel()

            # ============ SAT ============
            probs_SAT = torch.sigmoid(pred_SAT)
            preds_SAT = (probs_SAT >= 0.5).float()
            correct_train_SAT += (preds_SAT == label_train_SAT).sum().item()
            total_train_SAT += label_train_SAT.numel()

            # ============ PD1 ============
            probs_PD1 = torch.sigmoid(pred_PD1)
            preds_PD1 = (probs_PD1 >= 0.5).float()
            correct_train_PD1 += (preds_PD1 == label_train_PD1).sum().item()
            total_train_PD1 += label_train_PD1.numel()

            # ============ RA1 ============
            probs_RA1 = torch.sigmoid(pred_RA1)
            preds_RA1 = (probs_RA1 >= 0.5).float()
            correct_train_RA1 += (preds_RA1 == label_train_RA1).sum().item()
            total_train_RA1 += label_train_RA1.numel()

        train_loss = epoch_loss / len(train_loader)

        train_acc_ATC = correct_train_ATC / max(total_train_ATC, 1)
        train_acc_SAT = correct_train_SAT / max(total_train_SAT, 1)
        train_acc_PD1 = correct_train_PD1 / max(total_train_PD1, 1)
        train_acc_RA1 = correct_train_RA1 / max(total_train_RA1, 1)

        tensor_board_writer.add_scalar("Training/Loss", train_loss, epoch)
        tensor_board_writer.add_scalar("Train/Acc_ATC", train_acc_ATC, epoch)
        tensor_board_writer.add_scalar("Train/Acc_SAT", train_acc_SAT, epoch)
        tensor_board_writer.add_scalar("Train/Acc_PD1", train_acc_PD1, epoch)
        tensor_board_writer.add_scalar("Train/Acc_RA1", train_acc_RA1, epoch)
        
        

        # -------------------------------VALIDATION---------------------------
        trace_model.eval()
        val_loss = 0.0

        #========= Debugging ==========
        val_loss_ATC = 0.0
        val_loss_SAT = 0.0
        val_loss_PD1 = 0.0
        val_loss_RA1 = 0.0
        
        correct_val_ATC = 0
        correct_val_SAT = 0
        correct_val_PD1 = 0
        correct_val_RA1 = 0

        total_val_ATC = 0
        total_val_SAT = 0
        total_val_PD1 = 0
        total_val_RA1 = 0

        with torch.no_grad():
            for inputs_val, targets_val in validation_loader:

                label_val_ATC = targets_val["ATC"].unsqueeze(1)
                label_val_SAT = targets_val["SAT"].unsqueeze(1)
                label_val_PD1 = targets_val["PD1"].unsqueeze(1)
                label_val_RA1 = targets_val["RA1"].unsqueeze(1)

            
                delta_elapsed = get_elapsed_feature(inputs_val["timestamps"])
                delta_between = get_between_features(inputs_val["timestamps"])

                logits_val = trace_model(
                    inputs_val["aid"],
                    inputs_val["type"],
                    delta_elapsed,
                    delta_between
                )

                pred_ATC_val = logits_val[:, 0:1]
                pred_SAT_val = logits_val[:, 1:2]
                pred_PD1_val = logits_val[:, 2:3]
                pred_RA1_val = logits_val[:, 3:4]
                
                loss_ATC_val = criterion(pred_ATC_val, label_val_ATC.float())
                loss_SAT_val = criterion(pred_SAT_val, label_val_SAT.float())
                loss_PD1_val = criterion(pred_PD1_val, label_val_PD1.float())
                loss_RA1_val = criterion(pred_RA1_val, label_val_RA1.float())

                loss_validation = loss_ATC_val + loss_SAT_val + loss_PD1_val + loss_RA1_val
                
                val_loss += loss_validation.item()
                val_loss_ATC += loss_ATC_val.item()
                val_loss_SAT += loss_SAT_val.item()
                val_loss_PD1 += loss_PD1_val.item()
                val_loss_RA1 += loss_RA1_val.item()
                
                # ============ ATC ============
                probs_ATC_val = torch.sigmoid(pred_ATC_val)
                preds_ATC_val = (probs_ATC_val >= 0.5).float()
                correct_val_ATC += (preds_ATC_val == label_val_ATC).sum().item()
                total_val_ATC += label_val_ATC.numel()
                
                # ============ SAT ============
                probs_SAT_val = torch.sigmoid(pred_SAT_val)
                preds_SAT_val = (probs_SAT_val >= 0.5).float()
                correct_val_SAT += (preds_SAT_val == label_val_SAT).sum().item()
                total_val_SAT += label_val_SAT.numel()
                
                # ============ PD1 ============
                probs_PD1_val = torch.sigmoid(pred_PD1_val)
                preds_PD1_val = (probs_PD1_val >= 0.5).float()
                correct_val_PD1 += (preds_PD1_val == label_val_PD1).sum().item()
                total_val_PD1 += label_val_PD1.numel()

                # ============ RA1 ============
                probs_RA1_val = torch.sigmoid(pred_RA1_val)
                preds_RA1_val = (probs_RA1_val >= 0.5).float()
                correct_val_RA1 += (preds_RA1_val == label_val_RA1).sum().item()
                total_val_RA1 += label_val_RA1.numel()
                

        val_loss /= len(validation_loader)
        val_loss_ATC /= len(validation_loader)
        val_loss_SAT /= len(validation_loader)
        val_loss_PD1 /= len(validation_loader)
        val_loss_RA1 /= len(validation_loader)

        val_acc_ATC = correct_val_ATC / max(total_val_ATC, 1)
        val_acc_SAT = correct_val_SAT / max(total_val_SAT, 1)
        val_acc_PD1 = correct_val_PD1 / max(total_val_PD1, 1)
        val_acc_RA1 = correct_val_RA1 / max(total_val_RA1, 1)

        tensor_board_writer.add_scalar("Val/Loss", val_loss, epoch)
        tensor_board_writer.add_scalar("Val/Acc_ATC", val_acc_ATC, epoch)
        tensor_board_writer.add_scalar("Val/Acc_SAT", val_acc_SAT, epoch)
        tensor_board_writer.add_scalar("Val/Acc_PD1", val_acc_PD1, epoch)
        tensor_board_writer.add_scalar("Val/Acc_RA1", val_acc_RA1, epoch)
        
        ###Debugging        
        tensor_board_writer.add_scalar("Val/Loss_ATC", val_loss_ATC, epoch)
        tensor_board_writer.add_scalar("Val/Loss_SAT", val_loss_SAT, epoch)
        tensor_board_writer.add_scalar("Val/Loss_PD1", val_loss_PD1, epoch)
        tensor_board_writer.add_scalar("Val/Loss_RA1", val_loss_RA1, epoch)

        early_stopping(val_loss, trace_model)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

tensor_board_writer.close()

In [ ]:
#Load the Best model
early_stopping.load_best_weights(trace_model)

# Save all the model(Only to resume training)
torch.save({
    "epoch": epoch + 1,
    "model_state_dict": trace_model.state_dict(),
    "train_loss": train_loss,
    "val_loss": val_loss,
}, "checkpoint_TRACE.pt")